# Central Deflection of a Simply Supported Beam under a Central Point Load

This notebook derives, step by step, the mid-span deflection of a prismatic beam of flexural stiffness $EI$ and span $L$, pinned at both ends and carrying a single concentrated load $W$ at mid-span.

The derivation uses **Euler–Bernoulli beam theory** and the **double-integration method** [1, 2, 3]. The symbolic algebra is done with **SymPy** [5], so every step can be inspected and re-run. The result is then checked independently using **Castigliano's second theorem** [1, 6] and compared with standard published tables [1, 3, 4].

**Expected result:**

$$\delta_{\max} = \frac{W L^3}{48\,EI}$$

### Contents
1. Problem statement and assumptions
2. Governing equation and sign convention
3. Step 1 – Support reactions
4. Step 2 – Bending moment distribution
5. Step 3 – Integration and boundary conditions
6. Step 4 – Central deflection
7. Independent check – Castigliano's theorem
8. Worked numerical example and deflected shape
9. Discussion, limitations and references

**Requirements:** Python 3, `sympy`, `numpy`, `matplotlib` (e.g. `pip install jupyter sympy numpy matplotlib`).

## 1. Problem statement and assumptions

```
                     W
                     |
                     v
   A =================================== B
     ^                                 o
    /_\  <------ L/2 -----><--- L/2 --> /_\
    pin                             pin (roller)
   x = 0                              x = L
```

A beam $AB$ of span $L$ rests on a pinned support at $A$ ($x = 0$) and a pinned (roller) support at $B$ ($x = L$). Both supports stop vertical movement but let the beam rotate freely, so neither carries a moment. A point load $W$ acts downwards at mid-span, $x = L/2$.

**Assumptions** (standard Euler–Bernoulli assumptions [1, 2]):

- The material is homogeneous, isotropic and **linear elastic** (Hooke's law applies).
- The beam is **prismatic**: $EI$ is constant along its length.
- **Plane sections remain plane** and normal to the neutral axis, so shear deformation is ignored.
- Deflections and slopes are **small**, so the curvature is $\kappa \approx \mathrm{d}^2 v/\mathrm{d}x^2$.
- The beam's self-weight is ignored; the only load is $W$.

## 2. Governing equation and sign convention

In Euler–Bernoulli theory the curvature of the elastic curve is proportional to the bending moment [1, 2, 3]:

$$\kappa = \frac{1}{\rho} = \frac{M}{EI}$$

For small slopes, $\kappa \approx \dfrac{\mathrm{d}^2 v}{\mathrm{d}x^2}$. This gives the **differential equation of the elastic curve**:

$$EI\,\frac{\mathrm{d}^2 v}{\mathrm{d}x^2} = M(x)$$

**Sign convention** (as in Gere & Goodno [1]):

- $x$ is measured from the left support $A$.
- The deflection $v(x)$ is **positive upwards**, so a downward deflection is negative.
- A **sagging** bending moment (compression in the top fibres) is **positive**.

The *magnitude* of the downward deflection at mid-span is written $\delta = -v(L/2)$.

First we import the libraries and define the symbols. Declaring them `positive=True` lets SymPy simplify the algebra correctly.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math

sp.init_printing()

# Symbols: position x, span L, load W and flexural stiffness EI (treated as one constant)
x, L, W, EI = sp.symbols('x L W EI', positive=True)

# Unknown support reactions and integration constants
R_A, R_B = sp.symbols('R_A R_B')
C1, C2 = sp.symbols('C_1 C_2')

## 3. Step 1 – Support reactions

The beam is statically determinate, so the two vertical reactions $R_A$ and $R_B$ (positive upwards) come from the equilibrium equations alone [1, 3]:

- Vertical force equilibrium: $\sum F_y = 0 \;\Rightarrow\; R_A + R_B - W = 0$
- Moment equilibrium about $A$: $\sum M_A = 0 \;\Rightarrow\; R_B L - W \dfrac{L}{2} = 0$

There are no horizontal loads, so the horizontal reaction at the pin is zero.

In [ ]:
equilibrium = [
    sp.Eq(R_A + R_B - W, 0),        # sum of vertical forces = 0
    sp.Eq(R_B * L - W * L / 2, 0),  # sum of moments about A = 0
]
reactions = sp.solve(equilibrium, [R_A, R_B])

display(sp.Eq(R_A, reactions[R_A]), sp.Eq(R_B, reactions[R_B]))

As symmetry predicts, each support carries half the load: $R_A = R_B = W/2$.

## 4. Step 2 – Bending moment distribution

The point load causes a jump in the shear force at $x = L/2$, so the bending moment has a different expression on each side of the load. Because the beam and its loading are **symmetric about mid-span**, we only need to analyse the left half, $0 \le x \le L/2$. The right half is its mirror image.

Cut the beam at a section $x$ with $0 \le x \le L/2$ and take moments of the free body to the left of the cut. The only force on that free body is $R_A$, so

$$M(x) = R_A\,x = \frac{W x}{2}, \qquad 0 \le x \le \tfrac{L}{2}$$

This moment is sagging (positive). It rises linearly from zero at the support to its peak value $WL/4$ under the load [1, 3].

In [ ]:
M_left = reactions[R_A] * x   # bending moment for 0 <= x <= L/2
M_max = M_left.subs(x, L / 2) # peak moment, under the load

display(Math(r'M(x) = ' + sp.latex(M_left) + r'\quad (0 \le x \le L/2)'))
display(Math(r'M_{\max} = M(L/2) = ' + sp.latex(M_max)))

## 5. Step 3 – Integration and boundary conditions

Substitute $M(x)$ into the governing equation:

$$EI\,\frac{\mathrm{d}^2 v}{\mathrm{d}x^2} = \frac{W x}{2}$$

**First integration** gives the slope $\theta = \mathrm{d}v/\mathrm{d}x$:

$$EI\,\frac{\mathrm{d}v}{\mathrm{d}x} = \frac{W x^2}{4} + C_1$$

**Second integration** gives the deflection:

$$EI\,v = \frac{W x^3}{12} + C_1 x + C_2$$

The constants $C_1$ and $C_2$ come from two **boundary conditions** that hold on the left half [1, 2]:

1. **Support condition:** the pin at $A$ stops vertical movement, so $v(0) = 0$.
2. **Symmetry condition:** the deflected shape is symmetric about mid-span, so the elastic curve has a horizontal tangent there: $\left.\dfrac{\mathrm{d}v}{\mathrm{d}x}\right|_{x=L/2} = 0$.

> **Why not use $v(L) = 0$?** The expression $M = Wx/2$ only holds for $0 \le x \le L/2$, so it cannot be used for the support at $B$. Using symmetry avoids writing a second moment expression for the right half and enforcing slope and deflection continuity at the load. Macaulay's method [3, 6] is an alternative that handles the whole span with one expression.

In [ ]:
# Integrate EI v'' = M(x) twice, adding a constant each time
EI_slope = sp.integrate(M_left, x) + C1     # EI * dv/dx
EI_defl  = sp.integrate(EI_slope, x) + C2   # EI * v

display(Math(r"EI\,v'(x) = " + sp.latex(EI_slope)))
display(Math(r'EI\,v(x) = ' + sp.latex(EI_defl)))

# Boundary conditions: v(0) = 0 (pin at A) and v'(L/2) = 0 (symmetry)
bcs = [
    sp.Eq(EI_defl.subs(x, 0), 0),
    sp.Eq(EI_slope.subs(x, L / 2), 0),
]
constants = sp.solve(bcs, [C1, C2])
display(sp.Eq(C1, constants[C1]), sp.Eq(C2, constants[C2]))

Substituting the constants back gives the slope and the elastic curve for the left half of the beam:

$$\theta(x) = \frac{W}{16\,EI}\left(4x^2 - L^2\right), \qquad v(x) = -\frac{W x}{48\,EI}\left(3L^2 - 4x^2\right), \qquad 0 \le x \le \tfrac{L}{2}$$

These match the standard results in published tables [1, 3, 4].

In [ ]:
theta_left = sp.factor(EI_slope.subs(constants) / EI)  # slope dv/dx
v_left     = sp.factor(EI_defl.subs(constants) / EI)   # deflection v(x), positive upwards

display(Math(r'\theta(x) = ' + sp.latex(theta_left)))
display(Math(r'v(x) = ' + sp.latex(v_left)))

# Check: substituting back must satisfy the ODE and both boundary conditions
assert sp.simplify(EI * sp.diff(v_left, x, 2) - M_left) == 0
assert v_left.subs(x, 0) == 0
assert sp.simplify(theta_left.subs(x, L / 2)) == 0
print('ODE and boundary conditions satisfied.')

## 6. Step 4 – Central deflection

The largest deflection occurs where the slope is zero, which is at mid-span. Evaluating $v(x)$ at $x = L/2$:

$$v\!\left(\tfrac{L}{2}\right) = -\frac{W (L/2)}{48\,EI}\left(3L^2 - 4\cdot\tfrac{L^2}{4}\right) = -\frac{W L}{96\,EI}\cdot 2L^2 = -\frac{W L^3}{48\,EI}$$

The negative sign means the beam deflects **downwards**. The magnitude of the central deflection is

$$\boxed{\delta = \frac{W L^3}{48\,EI}}$$

The rotation at each support, which is the largest slope on the beam, is also useful:

$$\theta_A = -\theta_B = -\frac{W L^2}{16\,EI}$$

In [ ]:
delta   = sp.simplify(-v_left.subs(x, L / 2))  # downward deflection at mid-span
theta_A = sp.simplify(theta_left.subs(x, 0))   # slope at support A

display(Math(r'\delta = -v(L/2) = ' + sp.latex(delta)))
display(Math(r'\theta_A = ' + sp.latex(theta_A)))

assert sp.simplify(delta - W * L**3 / (48 * EI)) == 0

## 7. Independent check – Castigliano's second theorem

To confirm the result by a completely different method, we use **Castigliano's second theorem** [1, 6]. For a linear elastic structure, the displacement at the point where a load $P_i$ acts, measured in the direction of $P_i$, equals the partial derivative of the total strain energy $U$ with respect to that load:

$$\delta_i = \frac{\partial U}{\partial P_i}$$

If shear deformation is ignored (consistent with Euler–Bernoulli theory), the strain energy stored in bending is

$$U = \int_0^L \frac{M(x)^2}{2\,EI}\,\mathrm{d}x = 2\int_0^{L/2} \frac{1}{2\,EI}\left(\frac{W x}{2}\right)^2 \mathrm{d}x$$

The factor of 2 accounts for the identical right half. Differentiating $U$ with respect to $W$ gives the deflection under the load directly, with no boundary conditions needed.

In [ ]:
U = 2 * sp.integrate(M_left**2 / (2 * EI), (x, 0, L / 2))  # strain energy in bending
delta_castigliano = sp.simplify(sp.diff(U, W))

display(Math(r'U = ' + sp.latex(sp.simplify(U))))
display(Math(r'\delta = \frac{\partial U}{\partial W} = ' + sp.latex(delta_castigliano)))

assert sp.simplify(delta_castigliano - delta) == 0
print('Castigliano result agrees with the double-integration result.')

Both methods give the same result, $\delta = WL^3/(48EI)$. This agrees with the standard tabulated value for a simply supported beam with a central point load [1, 3, 4].

## 8. Worked numerical example and deflected shape

As an illustration, take a steel beam with these values (SI units throughout):

- Central point load: $W = 10$ kN
- Span: $L = 6$ m
- Young's modulus (structural steel): $E = 210$ GPa
- Second moment of area: $I = 1.0 \times 10^{-4}$ m$^4$

These values are for illustration only. In a real design, $I$ would come from section tables for the chosen member.

In [ ]:
W_val = 10e3     # N
L_val = 6.0      # m
E_val = 210e9    # Pa
I_val = 1.0e-4   # m^4
EI_val = E_val * I_val  # N m^2

params = {W: W_val, L: L_val, EI: EI_val}

delta_val = float(delta.subs(params))
theta_val = float(theta_A.subs(params))
M_max_val = float(M_max.subs(params))

print(f'Flexural stiffness EI       = {EI_val:.3e} N m^2')
print(f'Maximum bending moment      = {M_max_val / 1e3:.2f} kN m')
print(f'Central deflection delta    = {delta_val * 1e3:.3f} mm')
print(f'Support rotation theta_A    = {theta_val:.3e} rad')
print(f'Span / deflection ratio     = L / {L_val / delta_val:.0f}')

The plot below shows the deflected shape and the bending moment diagram over the full span. Values for the right half come from symmetry: $v(x) = v(L - x)$ for $L/2 \le x \le L$.

In [ ]:
v_func = sp.lambdify(x, v_left.subs(params), 'numpy')
M_func = sp.lambdify(x, M_left.subs(params), 'numpy')

xs = np.linspace(0, L_val, 401)
x_mirror = np.where(xs <= L_val / 2, xs, L_val - xs)  # use symmetry for the right half
v_vals = v_func(x_mirror)
M_vals = M_func(x_mirror)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.plot(xs, v_vals * 1e3, lw=2)
ax1.axhline(0, color='grey', lw=0.8)
ax1.plot(L_val / 2, -delta_val * 1e3, 'o')
ax1.annotate(rf'$\delta = WL^3/48EI$ = {delta_val * 1e3:.2f} mm',
             xy=(L_val / 2, -delta_val * 1e3), xytext=(0.5, -0.4 * delta_val * 1e3),
             arrowprops=dict(arrowstyle='->'))
ax1.set_ylabel('Deflection $v$ (mm)')
ax1.set_title('Deflected shape (upwards positive)')
ax1.grid(True, alpha=0.3)

ax2.fill_between(xs, M_vals / 1e3, alpha=0.3)
ax2.plot(xs, M_vals / 1e3, lw=2)
ax2.set_xlabel('Position along beam $x$ (m)')
ax2.set_ylabel('Bending moment $M$ (kN m)')
ax2.set_title('Bending moment diagram (sagging positive)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Discussion and limitations

- **Result:** for a pinned–pinned beam with a central point load, $\delta = \dfrac{WL^3}{48EI}$. The maximum moment is $WL/4$ and the support rotations are $\pm WL^2/(16EI)$.
- **Sensitivity:** the deflection scales with $L^3$, so doubling the span increases it eightfold. It is inversely proportional to $EI$, so choosing a deeper section (larger $I$) is the most effective way to reduce it.
- **Shear deformation:** Euler–Bernoulli theory ignores shear strain. For short, deep beams (span-to-depth ratio below about 10) the shear contribution can be significant, and Timoshenko beam theory should be used instead [2].
- **Small deflections:** the linearised curvature $\kappa \approx v''$ only holds when slopes are small, which is the case for typical structural beams under service loads.
- **Superposition:** the problem is linear, so this result can be added to the deflections from other load cases, such as self-weight as a uniformly distributed load, $5wL^4/(384EI)$ [1, 3].

## References

1. Gere, J.M. and Goodno, B.J. (2018). *Mechanics of Materials*, 9th ed. Boston: Cengage Learning. Chapter 9, *Deflections of Beams* (differential equation of the elastic curve, double integration, Castigliano's theorem), and the appendix tables of beam deflections and slopes.
2. Timoshenko, S.P. and Gere, J.M. (1972). *Mechanics of Materials*. New York: Van Nostrand Reinhold. Development of the elastic curve equation, and discussion of shear deformation effects.
3. Hibbeler, R.C. (2017). *Mechanics of Materials*, 10th ed. Pearson. Chapter 12, *Deflection of Beams and Shafts* (integration method, discontinuity functions), and the appendix on slopes and deflections of beams.
4. Young, W.C., Budynas, R.G. and Sadegh, A.M. (2012). *Roark's Formulas for Stress and Strain*, 8th ed. New York: McGraw-Hill. Table 8.1, *Shear, moment, slope, and deflection formulas for elastic straight beams*.
5. Meurer, A. et al. (2017). SymPy: symbolic computing in Python. *PeerJ Computer Science*, 3:e103. https://doi.org/10.7717/peerj-cs.103
6. Megson, T.H.G. (2019). *Structural and Stress Analysis*, 4th ed. Oxford: Butterworth-Heinemann. Energy methods (Castigliano's theorems) and Macaulay's method for beam deflections.